# 02 — Feature Engineering
**ZivaBasa MVP (Kaggle-Data Phase)**

This notebook takes the validated raw checkpoints from `01_data_acquisition_eda.ipynb` and builds
the feature set for each task head (Employment, Skills, Productivity), following the ChiedzaAI
feature taxonomy:

```
Raw → Ratio/Index → Interaction → (Learned, built later by the model trunk) → Fusion
```

**Input:** `data/raw/{employment,skills,productivity}_checked.parquet`
**Output:** `data/processed/{employment,skills,productivity}_features.parquet` + a shared
`data/processed/feature_dictionary.csv` documenting every engineered feature.

> Cleaning and encoding decisions here should trace back to the checklist you filled in at the
> end of notebook 01. If you skipped that, go confirm real column names before running this.


In [12]:
# --- Setup ---
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

pd.set_option("display.max_columns", 100)

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

def load_checkpoint(name):
    path = os.path.join(RAW_DIR, f"{name}_checked.parquet")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run 01_data_acquisition_eda.ipynb first.")
        return None
    df = pd.read_parquet(path)
    print(f"[{name}] loaded {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df

df_employment = load_checkpoint("employment")
df_skills = load_checkpoint("skills")
df_productivity = load_checkpoint("productivity")


[employment] loaded 3,000 rows x 25 cols
[skills] loaded 1,470 rows x 35 cols
[productivity] loaded 15,000 rows x 20 cols


## 1. Feature Dictionary (Living Document)

Every engineered feature gets logged here — name, category (raw/ratio/index/interaction),
source dataset, and the formula/derivation. This is what lets you swap in real bank data later
without guessing what each column means.


In [13]:
feature_dictionary = []

def log_feature(name, category, task_head, source_cols, description):
    feature_dictionary.append({
        "feature_name": name,
        "category": category,          # raw | ratio | index | interaction | temporal
        "task_head": task_head,         # employment | skills | productivity
        "source_columns": source_cols,
        "description": description,
    })


## 2. Missing Value Handling

Applied per-column, per-dataset. Default strategy: numeric → median impute with a `_was_missing`
flag; categorical → mode impute or an explicit "Unknown" category. Override per-column below
based on what Section 3 of notebook 01 flagged.


In [14]:
def handle_missing(df, name, numeric_strategy="median", categorical_strategy="mode"):
    if df is None:
        return None
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in num_cols:
        if df[col].isna().any():
            df[f"{col}_was_missing"] = df[col].isna().astype(int)
            fill_val = df[col].median() if numeric_strategy == "median" else df[col].mean()
            df[col] = df[col].fillna(fill_val)

    for col in cat_cols:
        if df[col].isna().any():
            fill_val = df[col].mode().iloc[0] if categorical_strategy == "mode" else "Unknown"
            df[col] = df[col].fillna(fill_val)

    print(f"[{name}] missing values handled. Remaining NaNs: {df.isna().sum().sum()}")
    return df

df_employment = handle_missing(df_employment, "employment")
df_skills = handle_missing(df_skills, "skills")
df_productivity = handle_missing(df_productivity, "productivity")


[employment] missing values handled. Remaining NaNs: 0
[skills] missing values handled. Remaining NaNs: 0
[productivity] missing values handled. Remaining NaNs: 0


## 3. Outlier Handling

Clip numeric columns at the 1st/99th percentile rather than dropping rows — preserves sample
size on already-small proxy datasets. Adjust `cols_to_clip` per dataset based on the skew values
you saw in notebook 01, Section 4.


In [15]:
def clip_outliers(df, cols, name, lower=0.01, upper=0.99):
    if df is None:
        return None
    df = df.copy()
    cols = [c for c in cols if c in df.columns]
    for col in cols:
        lo, hi = df[col].quantile(lower), df[col].quantile(upper)
        n_clipped = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lo, hi)
        if n_clipped:
            print(f"[{name}] clipped {n_clipped} outliers in '{col}'")
    return df

# Adjust to real numeric columns confirmed in notebook 01
df_employment = clip_outliers(df_employment, ["automation_risk", "risk_score", "salary"], "employment")
df_skills = clip_outliers(df_skills, ["MonthlyIncome", "YearsAtCompany", "TrainingTimesLastYear"], "skills")
df_productivity = clip_outliers(df_productivity, ["ai_adoption_level", "skill_gap_index"], "productivity")


[skills] clipped 30 outliers in 'MonthlyIncome'
[skills] clipped 13 outliers in 'YearsAtCompany'
[productivity] clipped 296 outliers in 'ai_adoption_level'
[productivity] clipped 298 outliers in 'skill_gap_index'


## 4. Raw Feature Selection

Explicitly select the raw columns that carry forward into modeling, per task head. Being
explicit here (rather than "keep everything") keeps the feature dictionary honest and avoids
accidentally leaking a target-adjacent column into the inputs.


In [16]:
# --- Adjust these lists to the real, confirmed column names from notebook 01 ---
RAW_COLS = {
    "employment": ["job_role", "industry", "automation_risk", "salary", "digital_skill_level"],
    "skills": ["Age", "JobRole", "Department", "TrainingTimesLastYear", "YearsAtCompany",
               "MonthlyIncome", "JobSatisfaction", "PerformanceRating", "Attrition"],
    "productivity": ["industry", "ai_adoption_level", "skill_gap_index", "salary_trend"],
}

def select_raw(df, cols, name):
    if df is None:
        return None
    cols = [c for c in cols if c in df.columns]
    missing = set(RAW_COLS[name]) - set(cols)
    if missing:
        print(f"[{name}] WARNING — requested columns not found, skipping: {missing}")
    for c in cols:
        log_feature(c, "raw", name, [c], "Direct source column, no transformation.")
    return df[cols].copy()

feat_employment = select_raw(df_employment, RAW_COLS["employment"], "employment")
feat_skills = select_raw(df_skills, RAW_COLS["skills"], "skills")
feat_productivity = select_raw(df_productivity, RAW_COLS["productivity"], "productivity")


[employment] WARNING — requested columns not found, skipping: {'automation_risk', 'digital_skill_level', 'salary'}
[productivity] WARNING — requested columns not found, skipping: {'salary_trend'}


## 5. Ratio / Index Features

Derived single-column-family metrics — e.g. Training Hours per Employee, Automation Exposure
Index. These map directly onto the "Ratio/Index" tier of the ChiedzaAI taxonomy.

Adjust formulas once real column names/scales are confirmed — the ones below are reasonable
starting points for each dataset's typical schema.


In [17]:
# --- Skills: Training intensity index ---
if feat_skills is not None and {"TrainingTimesLastYear", "YearsAtCompany"}.issubset(feat_skills.columns):
    feat_skills["training_intensity_index"] = (
        feat_skills["TrainingTimesLastYear"] / feat_skills["YearsAtCompany"].replace(0, 1)
    )
    log_feature("training_intensity_index", "index", "skills",
                ["TrainingTimesLastYear", "YearsAtCompany"],
                "Training events per year of tenure — proxy for ongoing skill investment.")

# --- Employment: Automation exposure index (normalized 0-1) ---
if feat_employment is not None and "automation_risk" in feat_employment.columns:
    col = feat_employment["automation_risk"]
    feat_employment["automation_exposure_index"] = (col - col.min()) / (col.max() - col.min() + 1e-9)
    log_feature("automation_exposure_index", "index", "employment",
                ["automation_risk"],
                "Min-max normalized automation risk score, 0-1 scale.")

# --- Productivity: AI adoption index (normalized 0-1) ---
if feat_productivity is not None and "ai_adoption_level" in feat_productivity.columns:
    col = feat_productivity["ai_adoption_level"]
    feat_productivity["ai_adoption_index"] = (col - col.min()) / (col.max() - col.min() + 1e-9)
    log_feature("ai_adoption_index", "index", "productivity",
                ["ai_adoption_level"],
                "Min-max normalized AI adoption level, 0-1 scale.")

print("Ratio/Index features added where source columns were available.")


Ratio/Index features added where source columns were available.


## 6. Interaction Features

Cross-products between features that the ChiedzaAI proposal specifically calls out — e.g.
Training Investment × Skill Readiness, AI Adoption × Employment Level. These are still
hand-built (as opposed to the model's own "learned" representations from the shared trunk).


In [18]:
# --- Skills: Training investment x tenure-adjusted satisfaction ---
if feat_skills is not None and {"training_intensity_index", "JobSatisfaction"}.issubset(feat_skills.columns):
    feat_skills["training_x_satisfaction"] = (
        feat_skills["training_intensity_index"] * feat_skills["JobSatisfaction"]
    )
    log_feature("training_x_satisfaction", "interaction", "skills",
                ["training_intensity_index", "JobSatisfaction"],
                "Training investment weighted by job satisfaction — proxy for skill-readiness likely to convert to retention.")

# --- Employment: Automation exposure x digital skill level (inverse relationship expected) ---
if feat_employment is not None and {"automation_exposure_index", "digital_skill_level"}.issubset(feat_employment.columns):
    feat_employment["exposure_x_digital_skill"] = (
        feat_employment["automation_exposure_index"] * feat_employment["digital_skill_level"]
    )
    log_feature("exposure_x_digital_skill", "interaction", "employment",
                ["automation_exposure_index", "digital_skill_level"],
                "Automation exposure weighted by digital skill level — higher digital skill may offset raw exposure.")

print("Interaction features added where source columns were available.")


Interaction features added where source columns were available.


## 7. Categorical Encoding

One-hot encode low-cardinality categoricals (job role, department, industry). For anything with
high cardinality, log a warning rather than silently exploding the feature space — revisit with
target encoding if that comes up on real bank data.


In [19]:
def encode_categoricals(df, name, max_cardinality=20):
    if df is None:
        return None
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    for col in cat_cols:
        n_unique = df[col].nunique()
        if n_unique > max_cardinality:
            print(f"[{name}] WARNING — '{col}' has {n_unique} categories, one-hot encoding will "
                  f"add {n_unique} columns. Consider target/frequency encoding instead.")
    df = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)
    print(f"[{name}] encoded {len(cat_cols)} categorical columns -> {df.shape[1]} total columns")
    return df

feat_employment = encode_categoricals(feat_employment, "employment")
feat_skills = encode_categoricals(feat_skills, "skills")
feat_productivity = encode_categoricals(feat_productivity, "productivity")


[employment] encoded 2 categorical columns -> 28 total columns
[skills] encoded 3 categorical columns -> 19 total columns
[productivity] encoded 1 categorical columns -> 10 total columns


## 8. Scaling

Standardize numeric features (zero mean, unit variance) — required for the neural network in
notebook 04, and doesn't hurt the tree-based baselines in notebook 03. Fit scalers here and save
them so the same transform can be applied consistently at inference time later.


In [21]:
import joblib

SCALER_DIR = "../models/scalers"
os.makedirs(SCALER_DIR, exist_ok=True)

def scale_numeric(df, name, exclude_cols=None):
    if df is None:
        return None
    exclude_cols = exclude_cols or []
    df = df.copy()
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

    if not num_cols:
        print(f"[{name}] no numeric columns to scale; skipping scaler fit.")
        return df

    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    joblib.dump(scaler, os.path.join(SCALER_DIR, f"{name}_scaler.pkl"))
    print(f"[{name}] scaled {len(num_cols)} numeric columns, scaler saved.")
    return df

# Exclude target columns from scaling — adjust once targets are finalized in notebook 03/04
feat_employment = scale_numeric(feat_employment, "employment", exclude_cols=["automation_risk"])
feat_skills = scale_numeric(feat_skills, "skills", exclude_cols=["Attrition"] if feat_skills is not None and "Attrition" in feat_skills.columns else [])
feat_productivity = scale_numeric(feat_productivity, "productivity", exclude_cols=["ai_adoption_level"])


[employment] no numeric columns to scale; skipping scaler fit.
[skills] scaled 8 numeric columns, scaler saved.
[productivity] scaled 2 numeric columns, scaler saved.


## 9. Target Variable Definition

Define the prediction target explicitly per task head before saving. This is the single most
important cell in this notebook to get right — everything downstream (baselines, multi-task NN,
SHAP) depends on these being correct and non-leaky.


In [22]:
# --- Employment: binary high-risk flag from automation_risk (adjust threshold as needed) ---
if feat_employment is not None and "automation_risk" in feat_employment.columns:
    threshold = feat_employment["automation_risk"].quantile(0.75)  # top quartile = "high risk"
    feat_employment["target_high_automation_risk"] = (
        feat_employment["automation_risk"] > threshold
    ).astype(int)
    log_feature("target_high_automation_risk", "target", "employment",
                ["automation_risk"], f"1 if automation_risk above 75th percentile ({threshold:.3f}).")
    print("Employment target distribution:\n", feat_employment["target_high_automation_risk"].value_counts())

# --- Skills: Attrition already binary (Yes/No) in IBM HR dataset — encode to 0/1 ---
if feat_skills is not None:
    attrition_cols = [c for c in feat_skills.columns if c.startswith("Attrition_")]
    if attrition_cols:
        feat_skills["target_attrition"] = feat_skills[attrition_cols[0]]
        log_feature("target_attrition", "target", "skills", ["Attrition"],
                    "1 if employee left (attrition), from one-hot encoded Attrition column.")
        print("Skills target distribution:\n", feat_skills["target_attrition"].value_counts())

# --- Productivity: regression target, keep ai_adoption_level (or its index) as-is ---
if feat_productivity is not None and "ai_adoption_index" in feat_productivity.columns:
    feat_productivity["target_ai_adoption"] = feat_productivity["ai_adoption_index"]
    log_feature("target_ai_adoption", "target", "productivity", ["ai_adoption_index"],
                "Regression target — normalized AI adoption level.")


Skills target distribution:
 target_attrition
False    1233
True      237
Name: count, dtype: int64


## 10. Save Processed Features + Feature Dictionary


In [23]:
def save_processed(df, name):
    if df is None:
        print(f"[{name}] nothing to save.")
        return
    out_path = os.path.join(PROCESSED_DIR, f"{name}_features.parquet")
    df.to_parquet(out_path, index=False)
    print(f"[{name}] saved -> {out_path}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")

save_processed(feat_employment, "employment")
save_processed(feat_skills, "skills")
save_processed(feat_productivity, "productivity")

fd_df = pd.DataFrame(feature_dictionary)
fd_path = os.path.join(PROCESSED_DIR, "feature_dictionary.csv")
fd_df.to_csv(fd_path, index=False)
print(f"\nFeature dictionary saved -> {fd_path} ({len(fd_df)} entries)")
display(fd_df)


[employment] saved -> ../data/processed\employment_features.parquet  (3,000 rows x 28 cols)
[skills] saved -> ../data/processed\skills_features.parquet  (1,470 rows x 20 cols)
[productivity] saved -> ../data/processed\productivity_features.parquet  (15,000 rows x 11 cols)

Feature dictionary saved -> ../data/processed\feature_dictionary.csv (19 entries)


,feature_name,category,task_head,source_columns,description
0,job_role,raw,employment,[job_role],"Direct source column, no transformation."
1,industry,raw,employment,[industry],"Direct source column, no transformation."
2,Age,raw,skills,[Age],"Direct source column, no transformation."
3,JobRole,raw,skills,[JobRole],"Direct source column, no transformation."
4,Department,raw,skills,[Department],"Direct source column, no transformation."
5,TrainingTimesLastYear,raw,skills,[TrainingTimesLastYear],"Direct source column, no transformation."
6,YearsAtCompany,raw,skills,[YearsAtCompany],"Direct source column, no transformation."
7,MonthlyIncome,raw,skills,[MonthlyIncome],"Direct source column, no transformation."
8,JobSatisfaction,raw,skills,[JobSatisfaction],"Direct source column, no transformation."
9,PerformanceRating,raw,skills,[PerformanceRating],"Direct source column, no transformation."


## 11. Summary — Carry Forward to Notebook 03

- [ ] Confirmed target variable per task head (Section 9) makes sense and isn't leaking future info
- [ ] Feature dictionary reviewed — every engineered feature has a clear rationale
- [ ] Categorical encoding didn't blow up dimensionality unexpectedly (check Section 7 warnings)
- [ ] Scalers saved to `models/scalers/` for consistent reuse at inference time
- [ ] Row counts per dataset noted — baseline models in notebook 03 will report metrics per
      task head separately, so uneven sample sizes across task heads should be expected and stated
